# Text-to-speech — the agent speaks

`text_to_speech` turns text into an audio file via a pluggable backend and returns its path with a `MEDIA:` tag a chat surface can play. The **Edge** backend is free and needs no key — so this whole notebook runs offline of any paid API.

| § | What you'll see |
|---|---|
| 1 | Setup — this repo on the path, `pip install edge-tts` |
| 2 | The tool directly — Edge (free), audio you can **play inline** |
| 3 | Voices + backends (Edge / OpenAI / ElevenLabs) |
| 4 | A real agent that chooses to speak |
| 5 | How it works |

## 1 · Setup

This repo on the path, and the free Edge backend installed.

In [ ]:
import sys
from pathlib import Path


def repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'shipit_agent' / '__init__.py').exists():
            return candidate
    raise RuntimeError(f'Could not locate the shipit_agent repo from {start}')


REPO = repo_root(Path.cwd())
sys.path.insert(0, str(REPO))

# Free, no-key backend — Microsoft Edge neural voices.
try:
    import edge_tts  # noqa: F401
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts'], check=True)

import shipit_agent
print('shipit-agent', shipit_agent.__version__)

## 2 · The tool directly — Edge (free)

No API key. The tool synthesises the text, saves an `.mp3`, and returns its path. In a notebook we can **play it inline**.

In [ ]:
from shipit_agent.tools.text_to_speech import TextToSpeechTool, available_providers
from shipit_agent.tools.base import ToolContext

print('backends available:', available_providers())   # ['edge'] with no keys set

tool = TextToSpeechTool(output_dir='/tmp/shipit_audio')
out = tool.run(ToolContext(prompt=''),
               text='Welcome aboard. Your shipit agent can now speak — with a free, offline voice.')
print(out.text)
print('bytes:', out.metadata['bytes'], '| format:', out.metadata['format'])

In [ ]:
from IPython.display import Audio
Audio(out.metadata['path'])   # ▶ press play

### Pick a voice

Edge ships hundreds of neural voices. Pass any of them as `voice`.

In [ ]:
out2 = tool.run(ToolContext(prompt=''),
                text='And here is a different voice, reading the same kind of line.',
                voice='en-GB-RyanNeural')
print(out2.metadata['provider'], out2.metadata['path'])
Audio(out2.metadata['path'])

## 3 · Backends

The backend is chosen: an explicit `provider=` wins, else the single available one, else a free-first preference (`edge` → `openai` → `elevenlabs`). Each is availability-gated, so the agent is only ever offered speech it can produce. To add a paid, higher-fidelity voice, set a key:

```bash
export OPENAI_API_KEY=...        # gpt-4o-mini-tts
export ELEVENLABS_API_KEY=...    # eleven_multilingual_v2
```
…and they appear in `available_providers()` automatically.

## 4 · A real agent that chooses to speak

Give an agent the tool and it calls `text_to_speech` itself. The LLM part needs a provider — this reads a **local, gitignored `.env`** for a Vertex key (`VERTEX_SA_KEY=/path/to/service-account.json`); the speech stays free via Edge. Skip this cell if you have no LLM key — §2 already proved the tool.

In [ ]:
import os, json as _json
from shipit_agent.llms.factory import load_env_file, build_llm_from_settings
load_env_file()
VKEY = os.environ.get('VERTEX_SA_KEY') or os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
if not VKEY:
    raise SystemExit('No LLM key — set VERTEX_SA_KEY in .env, or skip; the tool works in §2.')
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = VKEY
os.environ['VERTEXAI_PROJECT'] = _json.load(open(VKEY))['project_id']
os.environ.setdefault('VERTEXAI_LOCATION', 'us-central1')
llm = build_llm_from_settings({'provider': 'vertex', 'model': 'vertex_ai/gemini-2.5-flash'}, load_env=False)

from shipit_agent import Agent
agent = Agent(llm=llm, tools=[TextToSpeechTool(output_dir='/tmp/shipit_audio')],
              auto_use_skills=False, auto_project_memory=False, skill_source=None, max_iterations=4)
spoken = None
for ev in agent.stream("Say out loud, as audio: 'The quarterly report is ready.'"):
    p = ev.payload or {}
    if ev.type == 'tool_completed' and (p.get('metadata') or {}).get('media'):
        spoken = p['metadata']['media']
        print('agent produced audio:', spoken)
Audio(spoken) if spoken else print('no audio produced')

## 5 · How it works

1. `text_to_speech(text, voice?)` resolves a backend (`edge` free by default).
2. The backend returns audio bytes; the tool saves an `.mp3` to a run cache.
3. It returns the **path + a `MEDIA:<path>` tag** — the convention a send pipeline turns into a playable voice message.
4. The tool declares a `check_fn`, so with no backend installed/keyed it is **stripped from the toolset** — the agent is never offered speech it can't make.

Audio isn't something a model *sees*, so (unlike `image_generate`) the return is a file reference, not inline bytes.